In [ ]:
from huggingface_hub import hf_hub_download

review_path = hf_hub_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    repo_type="dataset",
    filename="raw/review_categories/Electronics.jsonl"
)
meta_path = hf_hub_download(
    repo_id="McAuley-Lab/Amazon-Reviews-2023",
    repo_type="dataset",
    filename="raw/meta_categories/meta_Electronics.jsonl"
)

raw/review_categories/Electronics.jsonl: reconstructing file:   0%|          |  0.00B / 22.6GB            

raw/review_categories/Electronics.jsonl: downloading bytes:           |  0.00B            

raw/meta_categories/meta_Electronics.jso(…): reconstructing file:   0%|          |  0.00B / 5.25GB            

raw/meta_categories/meta_Electronics.jso(…): downloading bytes:           |  0.00B            

1. Load meta into DuckDB, keep only spec-relevant fields

In [ ]:
!pip install duckdb

In [ ]:
import duckdb

con = duckdb.connect("amazon_electronics.duckdb")

con.execute(f"""
    CREATE OR REPLACE TABLE meta AS
    SELECT
        parent_asin,
        title,
        main_category,
        categories,
        features,
        description,
        details,        -- spec dict (dimensions, model no, etc.)
        price,
        average_rating,
        rating_number
    FROM read_ndjson_auto('{meta_path}', ignore_errors=true)
""")
print(con.execute("SELECT COUNT(*) FROM meta").fetchone())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(1610012,)


2. Filter useful reviews, then keep only the top-15 most useful per product

2.1 Setting temp_directory

In [ ]:
try:
    con.close()
except NameError:
    pass  # con didn't exist yet, fine

import os
if os.path.exists("amazon_electronics.duckdb"):
    os.remove("amazon_electronics.duckdb")

In [ ]:
import duckdb, os

os.makedirs('/content/duckdb_tmp', exist_ok=True)

# setting configurations
con = duckdb.connect("amazon_electronics.duckdb", config={
    "temp_directory": "/content/duckdb_tmp",
    "memory_limit": "30GB",
    "threads": "4",
    "max_temp_directory_size": "50GB",
})

con.execute("PRAGMA preserve_insertion_order=false")

In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   74G  163G  32% /


In [ ]:
con.execute(f"""
    CREATE OR REPLACE TABLE reviews_top AS
    WITH filtered AS (
        SELECT parent_asin, rating, title AS review_title, text AS review_text,
               helpful_vote, timestamp
        FROM read_ndjson_auto('{review_path}', ignore_errors=true)
        WHERE length(text) > 40        -- drop one-liners
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY parent_asin ORDER BY helpful_vote DESC
        ) AS rn
        FROM filtered
    )
    SELECT * EXCLUDE (rn) FROM ranked WHERE rn <= 15   -- top 15 reviews/product
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            52Gi       5.4Gi       1.4Gi       2.2Mi        46Gi        47Gi
Swap:             0B          0B          0B


3. Join into one product-level table

In [ ]:
con.execute("SHOW TABLES").fetchdf()

,name
0,reviews_top


In [ ]:
con.execute(f"""
    CREATE OR REPLACE TABLE meta AS
    SELECT
        parent_asin,
        title,
        main_category,
        categories,
        features,
        description,
        details,
        price,
        average_rating,
        rating_number
    FROM read_ndjson_auto('{meta_path}', ignore_errors=true)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
con.execute("""
    CREATE OR REPLACE TABLE product_docs AS
    SELECT
        m.parent_asin,
        m.title,
        m.main_category,
        m.features,
        m.description,
        m.details,
        m.price,
        m.average_rating,
        list(r.review_text ORDER BY r.helpful_vote DESC) AS top_reviews
    FROM meta m
    LEFT JOIN reviews_top r ON m.parent_asin = r.parent_asin
    GROUP BY ALL
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

4. Export to Parquet so the rest of your pipeline doesn't depend on DuckDB

In [ ]:
con.execute("COPY product_docs TO 'product_docs.parquet' (FORMAT PARQUET)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

5. Load and start cleaning

In [ ]:
import re
import json
import pandas as pd

In [ ]:
df = pd.read_parquet("product_docs.parquet")
reviews = df.explode("top_reviews").rename(columns={"top_reviews": "review_text"})
reviews = reviews.dropna(subset=["review_text"]).reset_index(drop=True)
reviews = reviews.drop_duplicates(subset=["parent_asin", "review_text"])
print(f"Reviews after explode + dedup: {len(reviews)}")

Reviews after explode + dedup: 7783429


In [ ]:
reviews.head()

,parent_asin,title,main_category,features,description,details,price,average_rating,review_text
0,B08KT3N7RK,Time Resistance Leather Backpack up to 15 in L...,None,[✔️ BACKPACK THAT MAKES A STATEMENT – did you ...,[],"[(Product Dimensions, ""5.12 x 11.42 x 15.75 in...",266.0,4.4,A great bag and best quality for money.<br />I...
1,B08KT3N7RK,Time Resistance Leather Backpack up to 15 in L...,None,[✔️ BACKPACK THAT MAKES A STATEMENT – did you ...,[],"[(Product Dimensions, ""5.12 x 11.42 x 15.75 in...",266.0,4.4,This is a beautiful item but is smaller than a...
2,B08KT3N7RK,Time Resistance Leather Backpack up to 15 in L...,None,[✔️ BACKPACK THAT MAKES A STATEMENT – did you ...,[],"[(Product Dimensions, ""5.12 x 11.42 x 15.75 in...",266.0,4.4,The bag is beautifully made and I Love it!! I ...
3,B08KW4LDLG,Acetaken StreamCam Stand -25'' Flexible Arm Cl...,All Electronics,[StreamCam Stand: Ball-Head adapter with 1/4''...,[],"[(Brand Name, ""Acetaken""), (Item Weight, ""1.21...",20.9,4.4,The sturdiness of this is what I like the most...
4,B08KW4LDLG,Acetaken StreamCam Stand -25'' Flexible Arm Cl...,All Electronics,[StreamCam Stand: Ball-Head adapter with 1/4''...,[],"[(Brand Name, ""Acetaken""), (Item Weight, ""1.21...",20.9,4.4,I had the issue where my old camera broke Afte...


5.1 Checking missing values

In [ ]:
cols_to_check = ["price", "average_rating", "features", "description", "details"]

missing_by_category = (
    df.groupby("main_category")[cols_to_check]
      .apply(lambda g: g.isna().mean() * 100)
      .round(1)
)
missing_by_category["n_products"] = df.groupby("main_category").size()
missing_by_category.sort_values("price", ascending=False)

,price,average_rating,features,description,details,n_products
main_category,,,,,,
Audible Audiobooks,100.0,0.0,0.0,0.0,0.0,1
Collectible Coins,100.0,0.0,0.0,0.0,0.0,5
Magazine Subscriptions,100.0,0.0,0.0,0.0,0.0,2
Unique Finds,100.0,0.0,0.0,0.0,0.0,1
Collectibles & Fine Art,100.0,0.0,0.0,0.0,0.0,1
Fire Phone,96.8,0.0,0.0,0.0,0.0,31
Movies & TV,88.0,0.0,0.0,0.0,0.0,75
Portable Audio & Accessories,87.8,0.0,0.0,0.0,0.0,8050
Software,87.4,0.0,0.0,0.0,0.0,388


In [ ]:
electronics_categories = [
    "Computers", "Cell Phones & Accessories", "Camera & Photo",
    "Portable Audio & Accessories", "Apple Products"
]
df_electronics = df[df["main_category"].isin(electronics_categories)]
print(f"The electronics dataset has {len(df_electronics)} records.")


The electronics dataset has 789923 records.


In [ ]:
df_electronics.head()

,parent_asin,title,main_category,features,description,details,price,average_rating,top_reviews
8,B08R55S2M7,CISSOOK Case for MacBook Pro 13 Inch 2020 Rele...,Computers,[This case is ONLY compatible with New MacBook...,[CISSOOK 2020 New Pro 13 inch Hard Shell Case ...,"[(Standing screen display size, ""4 Inches""), (...",NaN,3.9,"[This is beautiful, durable and fits perfectly..."
9,B08RB3Y68G,Neewer 2 Packs Upgraded 660 2.4G LED Video Lig...,Camera & Photo,[],[],"[(Package Dimensions, ""32.2 x 10 x 6.8 inches""...",NaN,4.5,[[[VIDEOID:55a8e161bfa72069550e12e495c78f7d]] ...
12,B08SCLFY82,Funnytree 6x6ft Class of 2022 Graduation Party...,Camera & Photo,[],[],"[(Product Dimensions, ""7.87 x 7.87 x 0.39 inch...",NaN,4.8,[Definitely great quality and value. It was gr...
14,B08W8TQW1S,Lenovo Legion 5 Desktop PC Computer | 10th Gen...,Computers,[],[],"[(Standing screen display size, ""17.94 Inches""...",771.89,3.5,"[PC was dead out of the box. Had ""no operating..."
15,B08W9C2DHB,CUK AORUS 15G by Gigabyte 15 Inch Gamer Notebo...,Computers,[Processor: Intel Core i7-10870H Eight Core Pr...,[],"[(Standing screen display size, ""15.6 Inches"")...",NaN,3.5,"[I love every single thing about this laptop, ..."


In [ ]:
# missing value checking
def is_empty_or_na(series: pd.Series) -> pd.Series:
    def _check(x):
        if x is None:
            return True
        if isinstance(x, list):
            return len(x) == 0
        try:
            return bool(pd.isna(x))
        except (TypeError, ValueError):
            return False  # non-list, non-scalar (e.g. dict) — treat as present
    return series.apply(_check)

# details is often a dict — treat an empty dict as missing too
def is_empty_or_na_dict(series: pd.Series) -> pd.Series:
    def _check(x):
        if x is None:
            return True
        if isinstance(x, dict):
            return len(x) == 0
        if isinstance(x, list):
            return len(x) == 0
        try:
            return bool(pd.isna(x))
        except (TypeError, ValueError):
            return False
    return series.apply(_check)

In [ ]:
cols_to_check = ["price", "average_rating", "features", "description", "details","top_reviews"]

def missing_pct_for_group(g: pd.DataFrame) -> pd.Series:
    out = {}
    for c in cols_to_check:
        if c == "details":
            out[c] = is_empty_or_na_dict(g[c]).mean() * 100
        else:
            out[c] = is_empty_or_na(g[c]).mean() * 100
    return pd.Series(out)

missing_by_subcat = (
    df_electronics.groupby("main_category")
    .apply(missing_pct_for_group)
    .round(1)
)
missing_by_subcat["n_products"] = df_electronics.groupby("main_category").size()
missing_by_subcat = missing_by_subcat.sort_values("n_products", ascending=False)

print("=== Missing/empty %% by sub-category (corrected) ===")
print(missing_by_subcat)

/tmp/ipykernel_2243/4291084361.py:9: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  return bool(pd.isna(x))


=== Missing/empty %% by sub-category (corrected) ===
                              price  average_rating  features  description  \
main_category                                                                
Computers                      71.2             0.0       0.0          0.0   
Camera & Photo                 67.4             0.0       0.0          0.0   
Cell Phones & Accessories      70.8             0.0       0.0          0.0   
Portable Audio & Accessories   87.8             0.0       0.0          0.0   
Apple Products                 70.4             0.0       0.0          0.0   

                              details  top_reviews  n_products  
main_category                                                   
Computers                         1.0          8.5      418868  
Camera & Photo                    1.3          8.2      223690  
Cell Phones & Accessories         0.9          8.3      138237  
Portable Audio & Accessories      2.2          7.9        8050  
Apple Prod

/tmp/ipykernel_2243/666455487.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(missing_pct_for_group)


In [ ]:
threshold = 60  # % missing above which price is treated as unreliable
unreliable_price_categories = missing_by_subcat[missing_by_subcat["price"] > threshold].index.tolist()
print(f"\nCategories with >{threshold}% missing price (treat price as unreliable):")
print(unreliable_price_categories)


Categories with >60% missing price (treat price as unreliable):
['Computers', 'Camera & Photo', 'Cell Phones & Accessories', 'Portable Audio & Accessories', 'Apple Products']


Because most of the price data is missing, we will focus on average_rating, feature, description and top_reviews

5.2 Data cleaning

In [ ]:
df_electronics = df_electronics.drop(columns=['price'])

In [ ]:
# detect rows with empty (specs + decription + reviews)
import numpy as np

def is_empty_list_or_na(x):
    if x is None:
        return True
    if isinstance(x, np.ndarray):
        return x.size == 0
    if isinstance(x, (list, dict)):
        return len(x) == 0
    try:
        result = pd.isna(x)
        if isinstance(result, np.ndarray):
            return result.size == 0 or result.all()
        return bool(result)
    except (TypeError, ValueError):
        return False

no_features = df_electronics["features"].apply(is_empty_list_or_na)
no_description = df_electronics["description"].apply(is_empty_list_or_na)
no_reviews = df_electronics["top_reviews"].apply(is_empty_list_or_na)

dead_rows_mask = no_features & no_description & no_reviews
print(f"{dead_rows_mask.sum()} rows have empty features AND description AND no reviews")

0 rows have empty features AND description AND no reviews


In [ ]:
len(df_electronics)

789923

In [ ]:
# find a row that actually has a video-tag review
sample_row = None
for reviews_list in df_electronics["top_reviews"]:
    if any("VIDEOID" in str(r) for r in reviews_list):
        sample_row = reviews_list
        break

print(sample_row)

['[[VIDEOID:mo9NBX7A0D63JK]]The Zoom H4N eats up batteries. When I am on longer video shoots and there are no AC outlets, this cable, coupled with a portable battery pack, is a great solution. The coiled cable is especially durable.', "Did what it was supposed to do.  Can't ask for anything more than that.", 'Not as pictured and very cumbersome the real connector is a combo head of a mini usb and the dc plug for the recorder on the flip-side. but is working and helps me while extended recording over about 3 day shoot 8hrs each but depends on the usb source you use', 'This little cable helps me keep my H4n alive throughout a whole wedding reception! Such a great invention honestly!', 'Amazing device, provides all of the power that I need to get the most accurate recordings from the H4N Zoom', 'hard to use .. but ok .. newer models might be better']


5.3 PII removal (data privacy)

In [ ]:
!pip install presidio_analyzer
!pip install presidio_anonymizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 74.3 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 50.0.1
    Uninstalling cryptography-50.0.1:
      Successfully uninstalled cryptography-50.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyopenssl 26.4.0 requires cryptography<51,>=49.0.0, but you have cryptography 48.0.1 which is incompatible.


In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

PII_ENTITIES = [
    "EMAIL_ADDRESS", "PHONE_NUMBER", "PERSON",
    "CREDIT_CARD", "LOCATION", "US_SSN", "IBAN_CODE"
]

def redact_pii(text: str) -> str:
    if not text or not str(text).strip():
        return text
    text = str(text)
    results = analyzer.analyze(text=text, entities=PII_ENTITIES, language="en")
    if not results:
        return text
    return anonymizer.anonymize(text=text, analyzer_results=results).text

def redact_review_list(reviews_list):
    if not isinstance(reviews_list, list):
        return reviews_list
    return [redact_pii(r) for r in reviews_list]

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import time

sample = df_electronics["top_reviews"].head(50)
start = time.time()
sample.apply(redact_review_list)
elapsed = time.time() - start
print(f"{elapsed:.1f}s for 50 products (~{50/elapsed:.1f} products/sec)")
print(f"Estimated total time: {(len(df_electronics)/ (50/elapsed)) / 60:.1f} minutes")

11.3s for 50 products (~4.4 products/sec)
Estimated total time: 2962.8 minutes


5.4 Construct a smaller subset of electronics product - direction 1

In [ ]:
electronics_categories2 = [
    "Cell Phones & Accessories",
    "Portable Audio & Accessories"
]
df_electronics2 = df[df["main_category"].isin(electronics_categories2)]
print(f"The electronics dataset has {len(df_electronics2)} records.")


The electronics dataset has 146287 records.


In [ ]:
df_electronics2.head()

,parent_asin,title,main_category,features,description,details,price,average_rating,top_reviews
19,B08ZDT69DQ,Live Laugh Golf Airpods Case Cover for Apple A...,Cell Phones & Accessories,[【Pretty & Elegant printings】We focus on quali...,"[Convenient to attach the Airpods to your bag,...","[(Best Sellers Rank, {""Electronics"":871700,""He...",NaN,3.6,[Got this for my boyfriend. The order took way...
27,B096RJ32FB,"4 Pack Air Tag Holder Case,Compatible Apple Ai...",Cell Phones & Accessories,[【Will NOT Fall Off】The AirTag case was precis...,[AirTag Case Compatible Apple Air Tag Keychain...,"[(Package Dimensions, ""5.43 x 4.13 x 1.18 inch...",NaN,4.3,[They are so cute and stylish and easy to find...
40,B09KH866S8,"Onamicit Key Finder,4 Pack Smart Bluetooth Tra...",Cell Phones & Accessories,[4 Key Finders：Our key finder is very convenie...,[This anti-lost device is a product based on t...,"[(Package Dimensions, ""4.17 x 3.39 x 0.79 inch...",18.40,2.8,[This device sets the alarm if I close the app...
45,B09PZHTYJY,SOLTECH Glossy Screen Protector Compatible wit...,Cell Phones & Accessories,[[Crystal Clean] This premium 9H HD tempered s...,"[Dodge Ram Screen Protector, This Premium Temp...","[(Package Dimensions, ""9.45 x 6.65 x 0.47 inch...",8.39,4.1,[Very hard to get on centered (at least for me...
48,B09SP8FNC3,Mektron Case for MacBook Pro 16 INCH M1 A2485 ...,Cell Phones & Accessories,[[Compatible with] - 2021 MacBook Pro 16 inch ...,[Mektron Crystal Hard Shell Case for MacBook P...,"[(Package Dimensions, ""17.8 x 11.38 x 1.46 inc...",28.99,5.0,"[Hi think this product is absolutely stunning,..."


In [ ]:
import pandas as pd
from google.colab import files

!pip install openpyxl

df_electronics2.to_excel('df_electronics2.xlsx', index=False)
files.download('df_electronics2.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("df_electronics2.xlsx", "/content/drive/MyDrive/df_electronics2.xlsx")

5.4 Construct a smaller subset - direction 2

In [ ]:
# handles None, NaN, empty list, empty dict, empty numpy array
def is_populated(x, min_list_len: int = 1) -> bool:
    if x is None:
        return False
    if isinstance(x, np.ndarray):
        x = x.tolist()
    if isinstance(x, (list, dict)):
        return len(x) >= min_list_len
    try:
        return not bool(pd.isna(x))
    except (TypeError, ValueError):
        return True  # unrecognized non-null type — treat as present

In [ ]:
# define full entry in the dataset
MIN_REVIEWS = 3          # at least this many usable reviews
MIN_TITLE_LEN = 5        # non-trivial title

has_title = df_electronics["title"].apply(lambda x: is_populated(x) and len(str(x).strip()) >= MIN_TITLE_LEN)
has_features = df_electronics["features"].apply(lambda x: is_populated(x))
has_description = df_electronics["description"].apply(lambda x: is_populated(x))
has_details = df_electronics["details"].apply(lambda x: is_populated(x))
has_reviews = df_electronics["top_reviews"].apply(lambda x: is_populated(x, min_list_len=MIN_REVIEWS))

In [ ]:
full_entry_mask = has_title & has_features & has_description & has_details & has_reviews

df_full = df_electronics[full_entry_mask].reset_index(drop=True)

print(f"Full entries: {len(df_full)} / {len(df_electronics)} "
      f"({len(df_full)/len(df_electronics)*100:.1f}%)")

Full entries: 198592 / 789923 (25.1%)


In [ ]:
completeness_by_category = (
    df_full.groupby("main_category").size()
    .rename("n_full")
    .to_frame()
    .join(df_electronics.groupby("main_category").size().rename("n_total"))
)
completeness_by_category["pct_full"] = (
    completeness_by_category["n_full"] / completeness_by_category["n_total"] * 100
).round(1)
print(completeness_by_category.sort_values("n_full", ascending=False))

                              n_full  n_total  pct_full
main_category                                          
Computers                     107818   418868      25.7
Camera & Photo                 55088   223690      24.6
Cell Phones & Accessories      34321   138237      24.8
Portable Audio & Accessories    1345     8050      16.7
Apple Products                    20     1078       1.9


In [ ]:
import pandas as pd
from google.colab import files

!pip install openpyxl

df_full.to_excel('df_full.xlsx', index=False)
files.download('df_full.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("df_full.xlsx", "/content/drive/MyDrive/df_full.xlsx")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/df_full.xlsx'

5.5 PII removal on dataset (direction 2)

In [ ]:
# check the time first
import time

sample = df_full["top_reviews"].head(50)
start = time.time()
sample.apply(redact_review_list)
elapsed = time.time() - start
print(f"{elapsed:.1f}s for 50 products (~{50/elapsed:.1f} products/sec)")
print(f"Estimated total time: {(len(df_full)/ (50/elapsed)) / 60:.1f} minutes")

13.6s for 50 products (~3.7 products/sec)
Estimated total time: 900.1 minutes


In [ ]:
import spacy
import numpy as np

nlp = spacy.load("en_core_web_sm", disable=["parser", "tagger", "lemmatizer"])

# 1. Flatten ALL reviews across ALL products into one big list
flat_texts = []
positions = []  # (row_index, review_index_within_list)

for idx, reviews_list in df_full["top_reviews"].items():
    if isinstance(reviews_list, np.ndarray):
        reviews_list = reviews_list.tolist()
    if isinstance(reviews_list, list):
        for i, r in enumerate(reviews_list):
            flat_texts.append(str(r) if r else "")
            positions.append((idx, i))

print(f"Total reviews to process: {len(flat_texts)}")

# 2. Time a slice first to sanity-check real throughput before committing
import time
test_slice = flat_texts[:5000]
start = time.time()
list(nlp.pipe(test_slice, batch_size=256))
elapsed = time.time() - start
rate = len(test_slice) / elapsed
print(f"{rate:.0f} docs/sec -> estimated total: {len(flat_texts)/rate/60:.1f} minutes")

Total reviews to process: 1824288
408 docs/sec -> estimated total: 74.5 minutes


In [ ]:
# to be runned
def redact_names(text, doc):
    redacted = text
    for ent in reversed(doc.ents):
        if ent.label_ == "PERSON":
            redacted = redacted[:ent.start_char] + "[PERSON]" + redacted[ent.end_char:]
    return redacted

redacted_flat = []
for text, doc in zip(flat_texts, nlp.pipe(flat_texts, batch_size=256)):
    redacted_flat.append(redact_names(text, doc))

# 3. Reassemble back into df_electronics["top_reviews"]
from collections import defaultdict
rebuilt = defaultdict(dict)
for (row_idx, review_idx), redacted_text in zip(positions, redacted_flat):
    rebuilt[row_idx][review_idx] = redacted_text

def rebuild_list(idx, original_list):
    if isinstance(original_list, np.ndarray):
        original_list = original_list.tolist()
    if idx not in rebuilt or not isinstance(original_list, list):
        return original_list
    return [rebuilt[idx].get(i, r) for i, r in enumerate(original_list)]

df_electronics["top_reviews"] = [
    rebuild_list(idx, lst) for idx, lst in df_electronics["top_reviews"].items()
]

Export the cleaned dataset

In [ ]:
import pandas as pd
from google.colab import files

!pip install openpyxl

df_electronics.to_excel('df_electronics.xlsx', index=False)
files.download('df_electronics.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy("df_electronics.xlsx", "/content/drive/MyDrive/df_electronics.xlsx")

Mounted at /content/drive


'/content/drive/MyDrive/df_electronics.xlsx'